# Self-Reflection: Rubric-Based, Evidence-Bound Revision

| Field | Value |
|---|---|
| Stage | Autonomous RAG patterns |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Reflection needs an explicit rubric, inspectable evidence, and a revision limit. A second generation is not automatically a better answer.

## 30-Second Summary

This notebook critiques an intentionally wrong retention answer against a source-backed rubric, performs one revision, and verifies factual value and citation. The loop cannot revise more than once.

## Why This Matters

Open-ended self-critique can rubber-stamp errors or loop indefinitely. Deterministic checks make the lesson's acceptance criteria visible.

## Scope

| Covers | Does not cover |
|---|---|
| Correctness/citation rubric, bounded revision, acceptance gate | LLM-as-judge calibration, style optimization, unlimited debate |


## Mental Model

```text
evidence -> draft -> rubric critique -> revise once -> verify -> accept/abstain
```


In [1]:
evidence = {"source": "retention", "text": "Audit logs are retained for thirty days."}
question = "How long are audit logs retained?"
MAX_REVISIONS = 1

def draft_answer() -> str:
    return "Audit logs are retained for ninety days [retention]."


## How It Works

The critic checks whether the evidence-backed value and source citation appear. Revision is allowed once and can only use the supplied evidence. Acceptance reruns the same rubric.


## Baseline

The initial draft is fluent and cited but contradicts the source, showing that citation presence alone is not faithfulness.


In [2]:
draft = draft_answer()
draft


'Audit logs are retained for ninety days [retention].'

## Technique Implementation

The rubric returns structured booleans and issue labels. The reviser copies the supported claim rather than adding new facts.


In [3]:
def critique(answer: str) -> dict:
    lower = answer.lower()
    return {
        "supported_value": "thirty" in lower and "ninety" not in lower,
        "citation_present": "[retention]" in lower,
        "issues": [
            issue for issue, failed in (
                ("unsupported retention value", not ("thirty" in lower and "ninety" not in lower)),
                ("missing citation", "[retention]" not in lower),
            ) if failed
        ],
    }

initial_critique = critique(draft)
revised = f"{evidence['text']} [{evidence['source']}]" if initial_critique["issues"] else draft
final_critique = critique(revised)
initial_critique, revised, final_critique


({'supported_value': False,
  'citation_present': True,
  'issues': ['unsupported retention value']},
 'Audit logs are retained for thirty days. [retention]',
 {'supported_value': True, 'citation_present': True, 'issues': []})

## Controlled Experiment

We compare rubric pass rate before and after exactly one revision and require the final answer to contain no unsupported duration.


In [4]:
def pass_rate(report: dict) -> float:
    return sum((report["supported_value"], report["citation_present"])) / 2

results = {
    "initial_pass_rate": pass_rate(initial_critique),
    "final_pass_rate": pass_rate(final_critique),
    "revisions": int(revised != draft),
    "final_answer": revised,
}
results


{'initial_pass_rate': 0.5,
 'final_pass_rate': 1.0,
 'revisions': 1,
 'final_answer': 'Audit logs are retained for thirty days. [retention]'}

## Evaluation

The initial answer passes citation presence but fails factual support (**0.5 rubric pass rate**). One evidence-bound revision reaches **1.0** and removes `ninety`. This rule-based judge is reliable only for the explicit fixture.


In [5]:
assert results["initial_pass_rate"] == 0.5 and results["final_pass_rate"] == 1.0
assert results["revisions"] == MAX_REVISIONS
assert "thirty" in revised.lower() and "ninety" not in revised.lower()
assert final_critique["issues"] == []
print("Self-reflection checks passed.")


Self-reflection checks passed.


## Decision Guide

| Need | Mechanism |
|---|---|
| Exact fact/citation | Deterministic evidence check |
| Nuanced quality | Calibrated judge plus human sample |
| Missing evidence | Retrieve/abstain, not revise prose |
| Repeated failure | Stop at revision budget |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Critic approves wrong answer | Weak rubric/shared bias | Grounded deterministic checks/human audit |
| Revision invents facts | Evidence not constrained | Evidence-only revision contract |
| Endless revisions | No budget | Hard iteration limit |
| Score improves by gaming wording | Proxy overfit | Diverse labels and blind review |


## Production Notes

### Observability
Log rubric version, issue labels, revision count, evidence IDs, acceptance, and latency/cost.

### Safety and Guardrails
Critique never grants permission to use new data or tools.

### Latency and Cost
Run cheap deterministic checks first and reserve model judges for unresolved cases.


## Practice

Add a completeness requirement and an answer that is correct but omits the citation.

## Recall

Toggle - Recall: What bounds reflection?
A rubric, evidence set, and revision limit.

Toggle - Recall: Why can a judge be wrong?
It may share model bias, miss domain rules, or reward surface form.

## Sources

- [Self-Refine](https://arxiv.org/abs/2303.17651)
- Repository-owned synthetic retention evidence

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the deterministic rubric fixture | Calibrate model-based judges against human labels |
